# 00 — Data Download

This notebook pulls Sentinel-2 and Sentinel-1 satellite imagery for the Cambridge A14 corridor from Google Earth Engine and exports it to Google Drive.

**Study area:** Cambridge A14 corridor (Cambourne to Waterbeach), ~30×30km  
**Epochs:** Summer composites for 2019, 2021, 2023, 2025  
**Sensors:** Sentinel-2 L2A (optical, 10 bands) + Sentinel-1 GRD (SAR, VV + VH)  
**Output:** GeoTIFF files saved to GEOL0069/Project/Data on Google Drive

In [1]:
# Install the Earth Engine Python package (already available in Colab but good to confirm)
import subprocess
subprocess.run(["pip", "install", "earthengine-api", "--quiet"])

CompletedProcess(args=['pip', 'install', 'earthengine-api', '--quiet'], returncode=0)

In [2]:
import ee

# Authenticate your Google account with Earth Engine
# A popup will appear asking you to log in and copy a token — follow the prompts
ee.Authenticate()

# Initialise with your Google Cloud project
ee.Initialize(project='geol0069-project')

print("Earth Engine initialised successfully")

Earth Engine initialised successfully


**Bounding Box:**

The bounding box covers the A14 corridor from Cambourne in the west to Waterbeach in the north-east, taking in the key transformation sites: Bourn Airfield, Bar Hill logistics cluster, and Waterbeach New Town.


In [3]:
# Define the Cambridge A14 corridor bounding box
# Coordinates: (longitude_min, latitude_min, longitude_max, latitude_max)
study_area = ee.Geometry.Rectangle([-0.20, 52.08, 0.25, 52.38])

print("Study area defined")
print("Approximate coverage: 30km west-east, 33km north-south")



Study area defined
Approximate coverage: 30km west-east, 33km north-south


**Cloud Masking Function**

Sentinel-2 images contain a QA60 band which flags cloudy and cirrus pixels. This function masks those pixels out before we take the median composite. We also divide by 10000 to convert from raw digital numbers to surface reflectance values (0–1 scale).

In [4]:
def mask_s2_clouds(image):
    """
    Masks clouds and cirrus in a Sentinel-2 image using the QA60 band.
    Also scales reflectance values from raw integers to 0-1 range.
    """
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus respectively
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Both flags should be zero (clear sky)
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    # Apply mask and scale reflectance to 0-1
    return image.updateMask(mask).divide(10000)


**Sentinel-2 Level2A Collection Cell:**

For each epoch we pull all available Sentinel-2 L2A images over the study area during June–August, apply cloud masking, and take the median pixel value across all available images. This gives us a single cloud-free composite image per epoch.

We select 10 bands:
- **10m bands:** B2 (blue), B3 (green), B4 (red), B8 (NIR)
- **20m bands:** B5, B6, B7, B8A (red edge + NIR), B11, B12 (SWIR)

The 20m bands will be resampled to 10m in the preprocessing notebook.

In [5]:
# Bands to export
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

# Define the four epochs as (year, label) pairs
epochs = [
    (2019, '2019'),
    (2021, '2021'),
    (2023, '2023'),
    (2025, '2025')
]

# Build one composite per epoch and store in a dictionary
s2_composites = {}

for year, label in epochs:
    composite = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(study_area)
        .filterDate(f'{year}-06-01', f'{year}-08-31') #SUMMER MONTHS ONLY
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
        .select(S2_BANDS)
        .median()
        .clip(study_area)
    )
    s2_composites[label] = composite
    print(f"Sentinel-2 composite created: {label}")

print("\nAll four Sentinel-2 composites ready")




Sentinel-2 composite created: 2019
Sentinel-2 composite created: 2021
Sentinel-2 composite created: 2023
Sentinel-2 composite created: 2025

All four Sentinel-2 composites ready


Each composite is exported as a GeoTIFF at 10m resolution to your Google Drive folder. These are large files so exports run as background tasks on the GEE servers — they will not appear in your Drive instantly. Check the Tasks tab in the GEE code editor (code.earthengine.google.com) to monitor progress.

Expect each file to be approximately 500MB–1GB depending on cloud cover in the final composite.

In [6]:
# Export each Sentinel-2 composite to Google Drive
for label, composite in s2_composites.items():
    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'S2_{label}_cambridge',
        folder='Data',          # subfolder inside GEOL0069/Project/
        fileNamePrefix=f'S2_{label}_cambridge',
        region=study_area,
        scale=10,               # 10m resolution
        crs='EPSG:32630',       # UTM zone 30N — standard for UK
        maxPixels=1e10,
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f"Export started: S2_{label}_cambridge — check GEE Tasks tab")

print("\nAll Sentinel-2 export tasks submitted")


Export started: S2_2019_cambridge — check GEE Tasks tab
Export started: S2_2021_cambridge — check GEE Tasks tab
Export started: S2_2023_cambridge — check GEE Tasks tab
Export started: S2_2025_cambridge — check GEE Tasks tab

All Sentinel-2 export tasks submitted


Sentinel-1 uses radar rather than optical light, so cloud cover does not affect it. We select images in Interferometric Wide (IW) swath mode with both VV and VH polarisations, using the ascending orbit pass for consistency across epochs.

We take the mean (rather than median) of available images, which is standard practice for SAR to reduce speckle noise.

In [7]:
# Build one Sentinel-1 composite per epoch
s1_composites = {}

for year, label in epochs:
    composite = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(study_area)
        .filterDate(f'{year}-06-01', f'{year}-08-31')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .mean()
        .clip(study_area)
    )
    s1_composites[label] = composite
    print(f"Sentinel-1 composite created: {label}")

print("\nAll four Sentinel-1 composites ready")



Sentinel-1 composite created: 2019
Sentinel-1 composite created: 2021
Sentinel-1 composite created: 2023
Sentinel-1 composite created: 2025

All four Sentinel-1 composites ready


Same process as Sentinel-2. These files are much smaller (roughly 100–200MB each) since we only have two bands.

In [8]:
# Export each Sentinel-1 composite to Google Drive
for label, composite in s1_composites.items():
    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'S1_{label}_cambridge',
        folder='Data',
        fileNamePrefix=f'S1_{label}_cambridge',
        region=study_area,
        scale=10,
        crs='EPSG:32630',
        maxPixels=1e10,
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f"Export started: S1_{label}_cambridge — check GEE Tasks tab")

print("\nAll Sentinel-1 export tasks submitted")
print("\nNext steps:")
print("1. Go to code.earthengine.google.com and click the Tasks tab (top right)")
print("2. You should see 8 tasks running — 4 Sentinel-2 and 4 Sentinel-1")
print("3. Each task takes roughly 5-20 minutes to complete")
print("4. When done, the GeoTIFFs will appear in GEOL0069/Project/Data on your Drive")

Export started: S1_2019_cambridge — check GEE Tasks tab
Export started: S1_2021_cambridge — check GEE Tasks tab
Export started: S1_2023_cambridge — check GEE Tasks tab
Export started: S1_2025_cambridge — check GEE Tasks tab

All Sentinel-1 export tasks submitted

Next steps:
1. Go to code.earthengine.google.com and click the Tasks tab (top right)
2. You should see 8 tasks running — 4 Sentinel-2 and 4 Sentinel-1
3. Each task takes roughly 5-20 minutes to complete
4. When done, the GeoTIFFs will appear in GEOL0069/Project/Data on your Drive
